**Libraries**

In [1]:
import warnings
warnings.filterwarnings('ignore') 
# Core Data Manipulation
import pandas as pd
import numpy as np
# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Preprocessing & Metrics
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    confusion_matrix, 
    classification_report
)

# The Champion Algorithm
from xgboost import XGBClassifier

# Deployment / Model Exporting
import joblib 

print("✅ Step 1: Libraries imported successfully.")

✅ Step 1: Libraries imported successfully.


**Import datasets**

In [2]:
# 2. IMPORT DATASETS
df_train = pd.read_csv('cargo_routing_train.csv')
df_val = pd.read_csv('cargo_routing_validation.csv')
df_test = pd.read_csv('cargo_routing_test.csv')
print(f"   -> Training Data:   {df_train.shape[0]} routes, {df_train.shape[1]} columns")
print(f"   -> Validation Data: {df_val.shape[0]} routes, {df_val.shape[1]} columns")
print(f"   -> Testing Data:    {df_test.shape[0]} routes, {df_test.shape[1]} columns")

   -> Training Data:   14000 routes, 21 columns
   -> Validation Data: 3500 routes, 21 columns
   -> Testing Data:    3500 routes, 21 columns


**Data cleaning and feature engineering**

In [3]:
# A. Feature Engineering: Extract timeline signals before dropping the timestamp
def extract_time_features(df):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['month'] = df['timestamp'].dt.month_name()       
    df['day_of_week'] = df['timestamp'].dt.day_name()   
    df['hour'] = df['timestamp'].dt.hour                
    return df

df_train = extract_time_features(df_train)
df_val = extract_time_features(df_val)
df_test = extract_time_features(df_test)


# B. Isolate the target and drop non-predictive administrative columns
cols_to_drop = ['record_id', 'timestamp', 'shipment_id', 'candidate_route_id']

# Separate Features (X) and Target (y) - PURE REGRESSION
# We keep the exact continuous 'optimization_score'
X_train = df_train.drop(columns=cols_to_drop + ['optimization_score'])
y_train = df_train['optimization_score']

X_val = df_val.drop(columns=cols_to_drop + ['optimization_score'])
y_val = df_val['optimization_score']

X_test = df_test.drop(columns=cols_to_drop + ['optimization_score'])
y_test = df_test['optimization_score']


# C. Secure Label Encoding for XGBoost
# UPDATED: Now includes our newly engineered 'month' and 'day_of_week' features!
categorical_features = [
    'origin', 'destination', 'cargo_type', 'priority', 
    'airline', 'season', 'shc_code', 'month', 'day_of_week'
]
print(f"   -> Encoding text features: {categorical_features}")

# Dictionary to store encoders for the deployment application
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    
    # Fit on ALL unique values combined to prevent unseen label crashes on Test/Val
    all_unique_values = pd.concat([X_train[col], X_val[col], X_test[col]]).astype(str).unique()
    le.fit(all_unique_values)
    
    # Transform AND force the data type to integer so XGBoost strictly accepts it
    X_train[col] = le.transform(X_train[col].astype(str)).astype(int)
    X_val[col] = le.transform(X_val[col].astype(str)).astype(int)
    X_test[col] = le.transform(X_test[col].astype(str)).astype(int)
    
    label_encoders[col] = le

print("   -> Data cleaning, timeline feature engineering, and encoding complete.")

   -> Encoding text features: ['origin', 'destination', 'cargo_type', 'priority', 'airline', 'season', 'shc_code', 'month', 'day_of_week']
   -> Data cleaning, timeline feature engineering, and encoding complete.


In [4]:
# ==========================================
# 5. MODEL TRAINING (XGBOOST REGRESSOR)
# ==========================================
import time
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

print("\n🧠 Step 5: Training the Champion XGBoost Regressor...")
start_time = time.time()

# Initialize the winning XGBoost architecture for our timeline-engineered data
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

# Train on historical data
xgb_model.fit(X_train, y_train)

# Convert seconds to minutes for display
training_time = round((time.time() - start_time) / 60, 2)
print(f"   -> Model training complete in {training_time} minutes.")

# ==========================================
# 6. VALIDATION PERFORMANCE METRICS
# ==========================================
print("\n📊 Step 6: Evaluating on Validation Data (Practice Exam)...")

# Predict ONLY on the validation set
y_pred_val = xgb_model.predict(X_val)

# Clip predictions between 0.0 and 1.0 to respect business logic boundaries
y_pred_val = np.clip(y_pred_val, 0.0, 1.0)

# Calculate exact regression metrics
r2_val = r2_score(y_val, y_pred_val)
mse_val = mean_squared_error(y_val, y_pred_val)
rmse_val = np.sqrt(mse_val)
mae_val = mean_absolute_error(y_val, y_pred_val)

# Display Results
print("============================================================")
print("🏆 XGBOOST VALIDATION METRICS")
print("============================================================")
print(f"R-Squared (R²) : {r2_val:.4f}")
print(f"RMSE           : {rmse_val:.4f}")
print(f"MAE            : {mae_val:.4f}")
print(f"MSE            : {mse_val:.4f}")
print("============================================================\n")


🧠 Step 5: Training the Champion XGBoost Regressor...
   -> Model training complete in 0.01 minutes.

📊 Step 6: Evaluating on Validation Data (Practice Exam)...
🏆 XGBOOST VALIDATION METRICS
R-Squared (R²) : 0.7517
RMSE           : 0.0797
MAE            : 0.0619
MSE            : 0.0064



**Testing on unseen data**

In [5]:
# ==========================================
# 7. MODEL TESTING & EVALUATION (UNSEEN DATA)
# ==========================================
print("\n🎯 Step 7: Evaluating on Unseen Test Data (The Final Audit)...")

# Predict the exact optimization scores for the Test dataset
y_pred_test = xgb_model.predict(X_test)

# Clip predictions between 0.0 and 1.0 to respect business logic boundaries
y_pred_test = np.clip(y_pred_test, 0.0, 1.0)

# Calculate final regression metrics
r2_final = r2_score(y_test, y_pred_test)
mse_final = mean_squared_error(y_test, y_pred_test)
rmse_final = np.sqrt(mse_final)
mae_final = mean_absolute_error(y_test, y_pred_test)

print("============================================================")
print("🏆 FINAL PRODUCTION METRICS (TEST DATA)")
print("============================================================")
print(f"R-Squared (R²) : {r2_final:.4f} ({(r2_final * 100):.2f}%)")
print(f"RMSE           : {rmse_final:.4f}")
print(f"MAE            : {mae_final:.4f}")
print(f"MSE            : {mse_final:.4f}")
print("============================================================\n")


# ==========================================
# 8. EXPORT MODEL FOR DEPLOYMENT
# ==========================================
print("💾 Step 8: Exporting Model and Encoders for Deployment...")

import joblib

# 1. Save the trained Champion XGBoost Model
joblib.dump(xgb_model, 'cargo_xgboost_model.pkl')
print("   -> Saved 'cargo_xgboost_model.pkl'")

# 2. Save the Label Encoders Dictionary
# (Crucial: Your deployment app needs this to translate text inputs like "Express", "Monday", or "August" into numbers)
joblib.dump(label_encoders, 'cargo_label_encoders.pkl')
print("   -> Saved 'cargo_label_encoders.pkl'")

print("\n✅ PIPELINE COMPLETE! Your upgraded timeline-aware model is ready for deployment.")


🎯 Step 7: Evaluating on Unseen Test Data (The Final Audit)...
🏆 FINAL PRODUCTION METRICS (TEST DATA)
R-Squared (R²) : 0.7426 (74.26%)
RMSE           : 0.0793
MAE            : 0.0607
MSE            : 0.0063

💾 Step 8: Exporting Model and Encoders for Deployment...
   -> Saved 'cargo_xgboost_model.pkl'
   -> Saved 'cargo_label_encoders.pkl'

✅ PIPELINE COMPLETE! Your upgraded timeline-aware model is ready for deployment.


In [7]:
# ==========================================
# ROUTE SCORE FORENSICS (EXPLAINING THE AI)
# ==========================================
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib

print("🔍 Loading Upgraded Model and Route Data...")

# 1. LOAD THE SAVED BRAIN
try:
    xgb_model = joblib.load('cargo_xgboost_model.pkl')
    label_encoders = joblib.load('cargo_label_encoders.pkl')
    df_test = pd.read_csv('cargo_routing_test.csv')
except FileNotFoundError:
    print("❌ ERROR: Could not find model or data files. Run the training script first.")
    exit()

# 2. SELECT A ROUTE TO INVESTIGATE (Let's look at the 900th route in the test set)
route_index = 100
route_raw = df_test.iloc[[route_index]].copy()

# Extract the requested raw business metrics for display
actual_score = route_raw['optimization_score'].values[0]
raw_cost = route_raw['cost_sar'].values[0]
raw_time = route_raw['total_transit_time_hours'].values[0]
raw_reliability = route_raw['reliability_score'].values[0]
raw_capacity = route_raw['capacity_available_kg'].values[0]
route_id = route_raw['record_id'].values[0]

# 3. PREPARE THE DATA FOR THE AI (NEW TIMELINE LOGIC)
# Extract our new timeline features on the fly before dropping the timestamp
route_raw['timestamp'] = pd.to_datetime(route_raw['timestamp'])
route_raw['month'] = route_raw['timestamp'].dt.month_name()
route_raw['day_of_week'] = route_raw['timestamp'].dt.day_name()
route_raw['hour'] = route_raw['timestamp'].dt.hour

cols_to_drop = ['record_id', 'timestamp', 'shipment_id', 'candidate_route_id', 'optimization_score']
X_single = route_raw.drop(columns=cols_to_drop)

# Keep a dictionary of the original human-readable text before encoding
human_readable_data = X_single.iloc[0].to_dict()

# Encode text to numbers using our saved encoders (INCLUDING NEW FEATURES)
categorical_features = [
    'origin', 'destination', 'cargo_type', 'priority', 
    'airline', 'season', 'shc_code', 'month', 'day_of_week'
]

for col in categorical_features:
    # Transform AND strictly cast to integer to prevent XGBoost object-type crashes
    X_single[col] = label_encoders[col].transform(X_single[col].astype(str)).astype(int)

# 4. EXTRACT THE EXACT MATHEMATICAL CONTRIBUTIONS
# Predict the score (Applying our business logic clipping)
raw_pred = xgb_model.predict(X_single)[0]
pred_score = max(0.0, min(1.0, float(raw_pred)))

# Extract the exact feature contributions using XGBoost's native SHAP matrix
# This returns an array where the last number is the baseline score, 
# and the rest are the exact mathematical +/- impacts of each feature.
dmatrix = xgb.DMatrix(X_single)
contributions = xgb_model.get_booster().predict(dmatrix, pred_contribs=True)[0]

base_score = contributions[-1]
feature_impacts = contributions[:-1]

# Calculate the total "push" (absolute sum of all movements) to figure out percentages
total_absolute_impact = np.sum(np.abs(feature_impacts))

# 5. PRINT THE FORENSIC REPORT
print("\n" + "="*65)
print(f"🔎 FORENSIC SCORE BREAKDOWN: {route_id}")
print("="*65)

print(f"🎯 AI Predicted Score : {pred_score:.4f} (Actual: {actual_score:.4f})")
print(f"🏗️ Base Starting Score: {base_score:.4f} (The average route score)\n")

print("📊 REQUESTED RAW METRICS:")
print("-" * 35)
print(f"Cost (SAR)          : {raw_cost:,.2f}")
print(f"Transit Time (hrs)  : {raw_time:.2f}")
print(f"Reliability         : {raw_reliability:.3f}")
print(f"Capacity Available  : {raw_capacity:,.2f} kg\n")

print("⚙️ EXACT FEATURE CONTRIBUTIONS (How we got from Base to Final):")
print("-" * 65)
print(f"{'Feature':<25} | {'Raw Value':<15} | {'Impact':<8} | {'% of Decision'}")
print("-" * 65)

# Build a list to sort features by how much they influenced the decision
impact_details = []
for i, col in enumerate(X_single.columns):
    impact = feature_impacts[i]
    perc = (abs(impact) / total_absolute_impact) * 100
    raw_val = human_readable_data[col]
    impact_details.append((col, raw_val, impact, perc))

# Sort by the most influential feature (highest percentage)
impact_details.sort(key=lambda x: x[3], reverse=True)

for col, raw_val, impact, perc in impact_details:
    # Format the impact string (e.g., +0.0452 or -0.1205)
    impact_str = f"+{impact:.4f}" if impact > 0 else f"{impact:.4f}"
    
    # Truncate raw values if they are long floats
    if isinstance(raw_val, float):
        raw_val_str = f"{raw_val:.2f}"
    else:
        raw_val_str = str(raw_val)
        
    print(f"{col:<25} | {raw_val_str:<15} | {impact_str:<8} | {perc:>5.1f}%")

print("-" * 65)
print("MATH CHECK: Base Score + Sum of Impacts = Predicted Score")
print(f"{base_score:.4f} + ({sum(feature_impacts):.4f}) = {base_score + sum(feature_impacts):.4f}")
print("=============================================================\n")

🔍 Loading Upgraded Model and Route Data...

🔎 FORENSIC SCORE BREAKDOWN: RTE_020897
🎯 AI Predicted Score : 0.9469 (Actual: 0.8780)
🏗️ Base Starting Score: 0.7577 (The average route score)

📊 REQUESTED RAW METRICS:
-----------------------------------
Cost (SAR)          : 57,459.49
Transit Time (hrs)  : 6.58
Reliability         : 0.880
Capacity Available  : 19,314.85 kg

⚙️ EXACT FEATURE CONTRIBUTIONS (How we got from Base to Final):
-----------------------------------------------------------------
Feature                   | Raw Value       | Impact   | % of Decision
-----------------------------------------------------------------
capacity_available_kg     | 19314.85        | +0.0904  |  28.1%
num_connections           | 0               | +0.0894  |  27.8%
reliability_score         | 0.88            | +0.0418  |  13.0%
cost_sar                  | 57459.49        | -0.0303  |   9.4%
cargo_weight_kg           | 8593.56         | -0.0219  |   6.8%
total_transit_time_hours  | 6.58         